# EXAONE-3.5-7.8B 회의 안건 생성 & 평가

> **수정 내용**: 모델 `main` 브랜치 원격 코드가 2026-02 Transformers v5 전용으로 갱신되어
> `transformers==4.43.3`과 충돌했습니다. 패치로 우회하는 대신 **4.43.3과 호환되는 최초 공개
> 리비전(`496aef0`)을 고정**해 원격 코드 패치 없이 정상 로드합니다.

In [1]:
# 호환 버전 설치 (실행 후 반드시 [커널 재시작] -> 다음 셀부터 다시 실행)
# transformers: 원격 코드 호환 위해 4.43.3 고정
# bitsandbytes: 런팟 CUDA(12.8)+최신 triton에 맞춰 최신으로 설치
#   (0.43.x 구버전은 libbitsandbytes_cuda128.so 없음 / triton.ops 에러)
!pip install -q transformers==4.43.3 accelerate==0.34.2
!pip install -q -U bitsandbytes

In [2]:
import gc
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f"transformers: {transformers.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    print(f"VRAM 전체: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

transformers: 4.43.3
GPU 사용 가능: True
GPU 이름: NVIDIA A40
VRAM 전체: 44.4 GB


In [ ]:
previous_meeting = """
[회의록] 2024년 11월 18일 (월) 스프린트 회고 회의
참석자: 김대표, 이CTO, 박PM, 최백엔드, 정프론트, 한디자이너

논의 내용
1. 스프린트 #12 회고
   - 회원가입/로그인 기능 완료, QA 통과
   - 대시보드 UI 3일 지연 (디자인 시안 수정 반복)
   - API 응답속도 800ms 초과, 최적화 필요
2. 투자 업데이트
   - 시리즈A 투자사 2곳 미팅 완료
   - MAU 지표와 리텐션 데이터 추가 요청
3. 팀 운영
   - 백엔드 개발자 1명 채용 결정
   - 온보딩 프로세스 문서화 미비

액션 아이템
- [이CTO] API 병목 구간 분석 보고서
- [박PM] 투자사 요청 지표 항목 정리
- [한디자이너] 대시보드 UI 확정 시안 공유
"""


unresolved_tasks = """
[미해결 태스크]
1. [높음] API 응답속도 최적화 - 담당: 최백엔드 (목표: 300ms)
2. [높음] 투자사 지표 대시보드 - 담당: 박PM+정프론트 (마감: 12월 5일)
3. [중간] 온보딩 플로우 개선 - 담당: 한디자이너+정프론트
4. [중간] 백엔드 개발자 채용 - 담당: 김대표+이CTO
5. [낮음] 고객 문의 대응 개선 - 담당: 박PM
"""

meeting_info = """
회의명: 12월 첫째 주 전체 팀 스탠드업
일시: 2024년 12월 2일 (월) 오전 10시
참석자: 전체 팀원 6명 / 예상 소요시간: 1시간
"""

print("회의 데이터 로드 완료")

더미 데이터 로드 완료


In [4]:
MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
# 핵심: main 코드는 Transformers v5 전용(2026-02)으로 갱신됨.
# 4.43.3과 호환되는 최초 공개 리비전을 고정해 원격 코드 패치 없이 로드.
REVISION = "496aef0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"로드 중: {MODEL_ID} (revision={REVISION})")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, trust_remote_code=True, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM 사용: {used:.2f} GB / {total:.1f} GB")
print("로드 완료")

로드 중: LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct (revision=496aef0)


Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

VRAM 사용: 4.92 GB / 44.4 GB
로드 완료


In [5]:
def generate(messages, max_new_tokens=600):
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

print("generate 함수 정의 완료")

generate 함수 정의 완료


In [ ]:
generation_messages = [
    {
        "role": "user",
        "content": (
            "당신은 회의 기초안건을 작성하는 전문 비서입니다. 주어진 자료를 분석하여 실용적이고 구체적인 회의 안건을 작성하세요.\n\n"
            "아래 자료를 바탕으로 회의 기초안건을 작성해주세요.\n\n"
            f"[회의 정보]\n{meeting_info}\n\n"
            f"[이전 회의록]\n{previous_meeting}\n\n"
            f"[미해결 태스크]\n{unresolved_tasks}\n\n"
            "위 자료를 종합하여 이번 회의에서 반드시 다뤄야 할 안건을 작성해주세요. "
            "각 안건에는 논의 목적과 주요 논의 포인트를 포함해주세요."
        )
    }
]

print("기초안건 생성 중...")
start = time.time()
generated_agenda = generate(generation_messages, max_new_tokens=600)
elapsed = time.time() - start

token_count = len(tokenizer.encode(generated_agenda))

print("\n" + "="*60)
print("[생성된 기초안건]")
print("="*60)
print(generated_agenda)
print("\n" + "="*60)
print(f"모델         : {MODEL_ID}")
print(f"생성 시간    : {elapsed:.2f}초")
print(f"생성 토큰 수 : {token_count}")
print(f"토큰/초      : {token_count / elapsed:.1f}")
print("="*60)

기초안건 생성 중...

[생성된 기초안건]
### 회의 기초 안건: 12월 첫째 주 전체 팀 스탠드업 (2024년 12월 2일 오전 10시)

#### 1. **스프린트 #13 시작 및 진행 상황 공유**
   - **논의 목적**: 스프린트 #13의 초기 계획 공유와 진행 상황 점검을 통해 팀의 목표 일치성을 확인하고, 초기 문제점을 미리 파악합니다.
   - **주요 논의 포인트**:
     - **회원가입/로그인 기능**: 완료 여부 및 향후 개선 사항
     - **대시보드 UI 개선**: 현재 진행 상황 및 최종 확정 예정일
     - **API 응답속도 최적화**: 진행 상황 및 목표 달성 여부 (최백엔드 담당)
     - **신규 기능 개발 계획**: 주요 기능 및 우선순위
     - **팀 내 역할 분배 및 협업 상태**: 각 팀원의 역할 확인 및 협업 효율성 검토

#### 2. **월간 서비스 지표 분석 및 개선 방안**
   - **논의 목적**: 11월 월간 서비스 지표를 분석하여 성과를 평가하고, 특히 이탈율과 고객 문의 응답 시간 개선을 위한 전략을 논의합니다.
   - **주요 논의 포인트**:
     - **MAU 및 리텐션**: 증가 추세 분석 및 향후 전략
     - **이탈율 개선**: 온보딩 플로우 개선 진행 상황 및 추가 개선 방안 (한디자이너+정프론트 담당)
     - **고객 문의 응답 시간**: 현재 평균 응답 시간 분석 및 단축 방안 (박PM 담당)
     - **경쟁사 동향**: 경쟁사 A의 베타 출시 계획에 따른 대응 전략

#### 3. **액션 아이템 진행 상황 점검 및 다음 단계 계획**
   - **논의 목적**: 이전 회의에서 할당된 액션 아이템의 진행 상황을 점검하고, 미완료 아이템에 대한 추가 조치 계획을 세웁니다.
   - **주요 논의 포인트**:
     - **[이CTO] API 병목 구간 분석 보고서**: 완료 여부 및 결과 공유
     - **[박PM] 투자사 요청 지표

In [ ]:
rubric = """
[평가 루브릭] 각 항목을 아래 기준에 따라 1~5점으로 채점

1) 안건 관련성
- 1점: 생성된 안건이 회의 주제와 전혀 무관함
- 2점: 일부 관련 있으나 절반 이상 주제 이탈
- 3점: 주제와 관련 있으나 핵심 안건 일부 누락
- 4점: 대부분 주제에 부합하며 경미한 이탈만 존재
- 5점: 모든 안건이 회의 주제와 정확히 일치함

2) 안건 구체성
- 1점: '논의 예정' 등 모호한 표현만 존재, 실행 불가
- 2점: 항목명만 있고 세부 내용 없음
- 3점: 일부 항목은 구체적이나 절반 이상 모호함
- 4점: 대부분 구체적이며 일부 보완 필요
- 5점: 모든 항목이 실행 가능한 수준으로 명확함

3) 외부자료 반영도 (외부자료 = 이전 회의록, 미해결 태스크)
- 1점: 외부자료가 전혀 반영되지 않음
- 2점: 외부자료 언급은 있으나 내용 반영 미흡
- 3점: 외부자료 일부 반영, 핵심 내용 누락
- 4점: 외부자료 주요 내용 반영, 일부 세부사항 누락
- 5점: 외부자료 핵심 내용이 안건에 완전히 반영됨

4) 항목 완결성
- 1점: 안건 항목이 1개 이하이거나 대부분 누락
- 2점: 주요 안건 절반 이상 누락
- 3점: 주요 안건 포함되나 세부 항목 누락 다수
- 4점: 대부분의 안건 포함, 1~2개 누락
- 5점: 모든 필요 안건이 빠짐없이 도출됨

5) 한국어 품질
- 1점: 문장이 어색하거나 비문/오탈자 다수
- 2점: 의미 전달은 되나 어색한 표현 다수
- 3점: 전반적으로 자연스러우나 일부 어색한 표현
- 4점: 자연스러운 한국어, 경미한 어색함만 존재
- 5점: 완전히 자연스럽고 격식에 맞는 한국어
"""

evaluation_messages = [
    {
        "role": "user",
        "content": (
            "당신은 회의 안건의 품질을 평가하는 전문가입니다. 주어진 루브릭의 점수 기준을 엄격히 적용하여 객관적으로 채점하세요.\n\n"
            "아래 회의 안건을 5가지 항목으로 평가해주세요.\n"
            "반드시 각 항목의 1~5점 기준에 비추어 채점하고, 그 점수를 준 이유를 한 줄로 설명하세요.\n\n"
            f"{rubric}\n"
            "[평가 대상 입력 자료]\n"
            f"- 회의 정보:\n{meeting_info}\n"
            f"- 이전 회의록:\n{previous_meeting}\n"
            f"- 미해결 태스크:\n{unresolved_tasks}\n\n"
            f"[평가할 안건]\n{generated_agenda}\n\n"
            "출력 형식(반드시 이 형식으로):\n"
            "1. 안건 관련성: X/5 - (이유)\n"
            "2. 안건 구체성: X/5 - (이유)\n"
            "3. 외부자료 반영도: X/5 - (이유)\n"
            "4. 항목 완결성: X/5 - (이유)\n"
            "5. 한국어 품질: X/5 - (이유)\n"
            "총점: X/25\n"
            "종합 의견:"
        )
    }
]

print("평가 중...")
evaluation_result = generate(evaluation_messages, max_new_tokens=700)

print("\n" + "="*60)
print("[평가 결과]")
print("="*60)
print(evaluation_result)

평가 중...

[평가 결과]
1. **안건 관련성**: 5/5 - 모든 안건이 회의 주제인 팀 스탠드업과 스프린트 진행 상황, 월간 서비스 지표 분석, 액션 아이템 점검 및 팀 운영 개선과 정확히 일치하며, 주요 회의 목표를 포괄적으로 다룹니다.

2. **안건 구체성**: 4/5 - 각 안건은 구체적인 논의 포인트와 담당 인원을 명시하고 있으며, 실행 가능한 수준으로 명확합니다. 그러나 일부 항목 (예: "신규 기능 개발 계획"의 세부 사항)에서 좀 더 구체적인 내용이 추가되면 더 높은 점수를 받을 수 있습니다.

3. **외부자료 반영도**: 4/5 - 이전 회의록과 미해결 태스크의 내용이 안건에 적절히 반영되어 있습니다. 특히 액션 아이템 진행 상황 점검과 미해결 태스크 업데이트 부분에서 외부자료의 핵심 내용이 잘 반영되어 있지만, 일부 세부사항의 보완이 필요할 수 있습니다.

4. **항목 완결성**: 4/5 - 주요 안건들이 대부분 포함되어 있으며, 팀 스탠드업의 핵심 요소들을 잘 다루고 있습니다. 다만, "팀 운영" 항목이 부분적으로만 작성되어 완전한 완결성을 갖추지 못한 점이 약간의 감점 요인입니다.

5. **한국어 품질**: 4/5 - 문장 구성이 대체로 자연스럽고 이해하기 쉬우나, 몇 가지 표현에서 약간의 어색함이 있습니다 (예: "팀 운영 및"의 불완전한 문장 끝맺음). 전반적으로 격식에 맞는 한국어로 작성되었지만, 미세한 수정이 필요할 수 있습니다.

**총점**: 21/25

**종합 의견**: 제시된 안건은 회의 주제와 매우 잘 연관되어 있으며, 구체적이고 실행 가능한 내용을 포함하고 있습니다. 외부자료의 반영도도 높아 회의의 연속성을 유지하는 데 효과적입니다. 항목 완결성과 한국어 표현에서 약간의 개선 여지가 있으나, 전반적으로 잘 구성된 안건입니다. 미세한 수정을 통해 더욱 완성도 높은 회의 안건으로 발전시킬 수 있을 것입니다.


In [8]:
print("="*60)
print(f"모델: {MODEL_ID}")
print("="*60)
print("\n[생성된 기초안건]")
print(generated_agenda)
print("\n[평가 결과]")
print(evaluation_result)

del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"\n모델 해제 완료 | 잔여 VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

모델: LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct

[생성된 기초안건]
### 회의 기초 안건: 12월 첫째 주 전체 팀 스탠드업 (2024년 12월 2일 오전 10시)

#### 1. **스프린트 #13 시작 및 진행 상황 공유**
   - **논의 목적**: 스프린트 #13의 초기 계획 공유와 진행 상황 점검을 통해 팀의 목표 일치성을 확인하고, 초기 문제점을 미리 파악합니다.
   - **주요 논의 포인트**:
     - **회원가입/로그인 기능**: 완료 여부 및 향후 개선 사항
     - **대시보드 UI 개선**: 현재 진행 상황 및 최종 확정 예정일
     - **API 응답속도 최적화**: 진행 상황 및 목표 달성 여부 (최백엔드 담당)
     - **신규 기능 개발 계획**: 주요 기능 및 우선순위
     - **팀 내 역할 분배 및 협업 상태**: 각 팀원의 역할 확인 및 협업 효율성 검토

#### 2. **월간 서비스 지표 분석 및 개선 방안**
   - **논의 목적**: 11월 월간 서비스 지표를 분석하여 성과를 평가하고, 특히 이탈율과 고객 문의 응답 시간 개선을 위한 전략을 논의합니다.
   - **주요 논의 포인트**:
     - **MAU 및 리텐션**: 증가 추세 분석 및 향후 전략
     - **이탈율 개선**: 온보딩 플로우 개선 진행 상황 및 추가 개선 방안 (한디자이너+정프론트 담당)
     - **고객 문의 응답 시간**: 현재 평균 응답 시간 분석 및 단축 방안 (박PM 담당)
     - **경쟁사 동향**: 경쟁사 A의 베타 출시 계획에 따른 대응 전략

#### 3. **액션 아이템 진행 상황 점검 및 다음 단계 계획**
   - **논의 목적**: 이전 회의에서 할당된 액션 아이템의 진행 상황을 점검하고, 미완료 아이템에 대한 추가 조치 계획을 세웁니다.
   - **주요 논의 포인트**:
     - **[이CTO] API 병목 구간 분석 보고서**: 완료 여부 및 결과